# LeetCode 3485 — Longest Common Prefix of K Strings After Removal bằng Python

- Link: https://leetcode.com/problems/longest-common-prefix-of-k-strings-after-removal/
- Độ khó: **Hard**
- Chủ đề: **Sắp xếp chuỗi** + LCP của `k` chuỗi liên tiếp, **max tiền tố/hậu tố**, **sparse table**; hoặc **Trie đếm** + "số nút tốt theo độ sâu"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def longestCommonPrefix(self, words: List[str], k: int) -> List[int]:
```

## 1. Mục tiêu học

1. Hiểu "LCP dài nhất của `k` chuỗi bất kỳ" = độ sâu lớn nhất của một tiền tố được **ít nhất `k` chuỗi** chia sẻ.
2. Dùng tính chất: sau khi **sắp xếp**, nhóm `k` chuỗi có LCP lớn nhất luôn có thể chọn là `k` chuỗi **liên tiếp**, và LCP của đoạn liên tiếp = LCP của **chuỗi đầu và chuỗi cuối**.
3. Xử lý "bỏ phần tử `i`": cửa sổ không chứa `i` giữ nguyên (max tiền tố/hậu tố), cửa sổ chứa `i` được thay bằng cửa sổ `k + 1` phần tử (truy vấn max đoạn).
4. Cách thứ hai bằng **Trie**: đếm số nút "tốt" (đếm ≥ `k`) ở mỗi độ sâu, khi bỏ một từ chỉ các nút có đếm **đúng bằng `k`** trên đường đi của nó bị mất.
5. Phân tích độ phức tạp theo **tổng độ dài** `S ≤ 10^5`.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` và số nguyên `k`. Với mỗi chỉ số `i`, bỏ phần tử thứ `i`, rồi tìm độ dài **tiền tố chung dài nhất** của **`k` chuỗi bất kỳ** (ở các chỉ số khác nhau) trong mảng còn lại. Nếu còn ít hơn `k` chuỗi, đáp án là `0`.

**Ví dụ 1:** `words = ["jump","run","run","jump","run"], k = 2` → `[3,4,4,3,4]`.

**Ví dụ 2:** `words = ["dog","racer","car"], k = 2` → `[0,0,0]`.

**Ràng buộc:** `1 ≤ k ≤ n ≤ 10^5`, `1 ≤ |words[i]| ≤ 10^4`, tổng độ dài `≤ 10^5`, chữ thường.

In [1]:
from typing import List
from collections import Counter

examples = [
    (["jump", "run", "run", "jump", "run"], 2, [3, 4, 4, 3, 4]),
    (["dog", "racer", "car"], 2, [0, 0, 0]),
]
for w, k, exp in examples:
    print(w, 'k =', k, '-> kỳ vọng', exp)

['jump', 'run', 'run', 'jump', 'run'] k = 2 -> kỳ vọng [3, 4, 4, 3, 4]
['dog', 'racer', 'car'] k = 2 -> kỳ vọng [0, 0, 0]


## 3. Kiến thức nền

### 3.1. LCP của hai chuỗi

In [2]:
def lcp(a, b):
    m = min(len(a), len(b))
    i = 0
    while i < m and a[i] == b[i]:
        i += 1
    return i

print(lcp('jump', 'jumper'), lcp('run', 'jump'), lcp('abc', 'abd'))

4 0 2


### 3.2. LCP của một nhóm = độ dài tiền tố chung

`k` chuỗi có tiền tố chung độ dài `d` ⇔ cả `k` chuỗi cùng có tiền tố `P` với `|P| = d`. Vậy đáp án (không bỏ gì) là `max{|P| : số chuỗi có tiền tố P ≥ k}`.

### 3.3. Sau khi sắp xếp, LCP của đoạn liên tiếp = LCP(đầu, cuối)

Trong danh sách đã sắp `s[0] ≤ s[1] ≤ …`, các chuỗi có cùng tiền tố `P` nằm **liên tiếp**. Và `lcp(s[j..t]) = lcp(s[j], s[t])` vì mọi chuỗi nằm giữa đều "kẹp" giữa hai chuỗi đầu và cuối.

In [3]:
s = sorted(["jump", "run", "run", "jump", "run", "jumbo", "rune"])
print(s)

def lcp_group(group):
    return min(lcp(group[0], g) for g in group)

for j in range(len(s) - 2):
    print(s[j:j + 3], 'lcp nhóm =', lcp_group(s[j:j + 3]), '| lcp(đầu, cuối) =', lcp(s[j], s[j + 2]))

['jumbo', 'jump', 'jump', 'run', 'run', 'run', 'rune']
['jumbo', 'jump', 'jump'] lcp nhóm = 3 | lcp(đầu, cuối) = 3
['jump', 'jump', 'run'] lcp nhóm = 0 | lcp(đầu, cuối) = 0
['jump', 'run', 'run'] lcp nhóm = 0 | lcp(đầu, cuối) = 0
['run', 'run', 'run'] lcp nhóm = 3 | lcp(đầu, cuối) = 3
['run', 'run', 'rune'] lcp nhóm = 3 | lcp(đầu, cuối) = 3


### 3.4. Sparse table cho truy vấn max đoạn

Xây `O(m log m)`, truy vấn `O(1)` bằng hai khối lũy thừa 2 phủ đoạn.

In [4]:
class SparseMax:
    def __init__(self, a):
        self.t = [list(a)]
        j = 1
        while (1 << j) <= len(a):
            prev, h = self.t[-1], 1 << (j - 1)
            self.t.append([max(prev[i], prev[i + h]) for i in range(len(a) - (1 << j) + 1)])
            j += 1

    def query(self, l, r):            # max a[l..r], trả 0 nếu đoạn rỗng
        if l > r:
            return 0
        j = (r - l + 1).bit_length() - 1
        return max(self.t[j][l], self.t[j][r - (1 << j) + 1])

sp = SparseMax([3, 1, 4, 1, 5, 9, 2, 6])
print(sp.query(0, 3), sp.query(2, 6), sp.query(5, 4))

4 9 0


### 3.5. Trie có đếm

Mỗi nút của trie lưu `cnt` = số từ đi qua nút đó. Nút ở độ sâu `d` có `cnt ≥ k` ⇔ có `k` từ chung tiền tố độ dài `d`.

In [5]:
def build_trie(words):
    children, cnt, depth = [{}], [0], [0]
    paths = []
    for w in words:
        u, path = 0, []
        for c in w:
            v = children[u].get(c)
            if v is None:
                v = len(children)
                children[u][c] = v
                children.append({}); cnt.append(0); depth.append(depth[u] + 1)
            u = v
            cnt[u] += 1
            path.append(u)
        paths.append(path)
    return children, cnt, depth, paths

ch, cnt, dep, paths = build_trie(examples[0][0])
print('số nút:', len(ch), '| cnt theo đường đi của "jump":', [cnt[u] for u in paths[0]])
print('cnt theo đường đi của "run":', [cnt[u] for u in paths[1]])

số nút: 8 | cnt theo đường đi của "jump": [2, 2, 2, 2]
cnt theo đường đi của "run": [3, 3, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1 (`k = 2`): các tiền tố và số lần xuất hiện: `j, ju, jum, jump` mỗi cái 2 lần; `r, ru, run` mỗi cái 3 lần.

- Bỏ một `"jump"` (i = 0 hoặc 3): `jump` còn 1 lần ⇒ không đủ; `run` còn 3 ⇒ đáp án 3.
- Bỏ một `"run"`: `run` còn 2 ≥ 2, nhưng `jump` vẫn 2 lần và dài hơn ⇒ đáp án 4.

Nhận xét: bỏ một từ chỉ làm **giảm 1** đếm của các tiền tố **của chính từ đó**. Tiền tố nào có đếm **đúng bằng `k`** và nằm trên từ bị bỏ thì mất tư cách. Tiền tố có đếm `> k` vẫn giữ.

In [6]:
def prefix_counts(words):
    c = Counter()
    for w in words:
        for L in range(1, len(w) + 1):
            c[w[:L]] += 1
    return c

print(dict(prefix_counts(examples[0][0])))

{'j': 2, 'ju': 2, 'jum': 2, 'jump': 2, 'r': 3, 'ru': 3, 'run': 3}


## 5. Cách 1 — Brute Force: đếm lại mọi tiền tố sau mỗi lần bỏ

In [7]:
def lcpk_brute(words, k):
    n = len(words)
    res = []
    for i in range(n):
        c = prefix_counts(words[:i] + words[i + 1:])
        res.append(max([len(p) for p, v in c.items() if v >= k], default=0))
    return res

for w, k, exp in examples:
    print(lcpk_brute(w, k), '| kỳ vọng', exp)

[3, 4, 4, 3, 4] | kỳ vọng [3, 4, 4, 3, 4]
[0, 0, 0] | kỳ vọng [0, 0, 0]


**Độ phức tạp:** mỗi lần bỏ tốn `O(Σ |w|²)` (cắt chuỗi tiền tố) ⇒ tổng `O(n · Σ|w|²)` — chỉ dùng làm oracle với dữ liệu nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

**Cách nhìn theo sắp xếp.** Sắp `words` thành `s`. Gọi

- `A[j] = lcp(s[j], s[j + k − 1])` — giá trị của cửa sổ `k` phần tử bắt đầu tại `j`;
- `B[j] = lcp(s[j], s[j + k])` — giá trị cửa sổ `k + 1` phần tử bắt đầu tại `j`.

Không bỏ gì: đáp án `= max A`. Khi bỏ phần tử ở vị trí `p` (trong `s`):

1. Các cửa sổ `k` phần tử **không chứa `p`**: `j ≤ p − k` hoặc `j ≥ p + 1` ⇒ max tiền tố/hậu tố của `A`.
2. Các nhóm `k` phần tử **liên tiếp sau khi bỏ `p`** mà "vắt qua" `p`: chính là cửa sổ `k + 1` phần tử `[j, j + k]` chứa `p`, bỏ `p` ra; LCP vẫn là `lcp(s[j], s[j + k])` = `B[j]` (nếu `p` là đầu/cuối cửa sổ thì nhóm còn lại nằm gọn trong cửa sổ, LCP ≥ `B[j]`, và đã được tính ở mục 1). ⇒ `max B[j]` với `j ∈ [p − k, p]` — truy vấn max đoạn.

**Cách nhìn theo trie.** `good[d]` = số nút ở độ sâu `d` có `cnt ≥ k`. Khi bỏ từ `w`, nút trên đường đi của `w` ở độ sâu `d` mất tư cách ⇔ `cnt = k`. Đáp án = độ sâu lớn nhất `d` với `good[d] − [nút độ sâu d của w có cnt = k] > 0`. Vì mỗi nút tốt có cha tốt, chỉ cần dò từ `maxD` xuống, và chỉ dò khi `d ≤ |w|`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Sắp xếp + cửa sổ `k` và `k + 1` + sparse table

In [8]:
def lcpk_sorted(words, k):
    n = len(words)
    if n - 1 < k:
        return [0] * n
    order = sorted(range(n), key=lambda i: words[i])
    s = [words[i] for i in order]
    pos = [0] * n
    for p, i in enumerate(order):
        pos[i] = p
    A = [lcp(s[j], s[j + k - 1]) for j in range(n - k + 1)]
    B = [lcp(s[j], s[j + k]) for j in range(n - k)]
    pre = [0] * (len(A) + 1)                 # pre[t] = max A[0..t-1]
    for j, x in enumerate(A):
        pre[j + 1] = max(pre[j], x)
    suf = [0] * (len(A) + 1)                 # suf[t] = max A[t..]
    for j in range(len(A) - 1, -1, -1):
        suf[j] = max(suf[j + 1], A[j])
    spB = SparseMax(B)
    res = []
    for i in range(n):
        p = pos[i]
        best = max(pre[max(0, p - k + 1)], suf[min(len(A), p + 1)])
        best = max(best, spB.query(max(0, p - k), min(p, len(B) - 1)))
        res.append(best)
    return res

for w, k, exp in examples:
    print(lcpk_sorted(w, k), '| kỳ vọng', exp)

[3, 4, 4, 3, 4] | kỳ vọng [3, 4, 4, 3, 4]
[0, 0, 0] | kỳ vọng [0, 0, 0]


### 7.2. Cách 3 — Trie đếm + số nút tốt theo độ sâu

In [9]:
def lcpk_trie(words, k):
    n = len(words)
    if n - 1 < k:
        return [0] * n
    children, cnt, depth, paths = build_trie(words)
    maxlen = max(map(len, words))
    good = [0] * (maxlen + 1)
    for u in range(1, len(cnt)):
        if cnt[u] >= k:
            good[depth[u]] += 1
    maxD = max((d for d in range(maxlen + 1) if good[d] > 0), default=0)
    res = []
    for path in paths:
        d = maxD
        while d > 0:
            lose = 1 if d <= len(path) and cnt[path[d - 1]] == k else 0
            if good[d] - lose > 0:
                break
            d -= 1
        res.append(d)
    return res

for w, k, exp in examples:
    print(lcpk_trie(w, k), '| kỳ vọng', exp)

[3, 4, 4, 3, 4] | kỳ vọng [3, 4, 4, 3, 4]
[0, 0, 0] | kỳ vọng [0, 0, 0]


Vì sao vòng `while` không chậm: nếu `maxD > |w|` thì `lose = 0` và `good[maxD] > 0` ⇒ dừng ngay. Nếu không, số bước ≤ `|w|`. Tổng mọi từ `≤ S`.

### 7.3. Cách 4 — Băm tiền tố (rolling hash) thay cho trie

Thay mỗi nút trie bằng **giá trị băm của tiền tố** (đếm bằng `dict`). Logic y hệt cách 3. Dùng băm đa thức mod số nguyên tố lớn kèm độ dài làm khóa để giảm va chạm (về lý thuyết vẫn có xác suất va chạm rất nhỏ).

In [10]:
def lcpk_hash(words, k, base=911382323, mod=(1 << 61) - 1):
    n = len(words)
    if n - 1 < k:
        return [0] * n
    cnt = Counter()
    keys = []
    for w in words:
        h, ks = 0, []
        for c in w:
            h = (h * base + ord(c)) % mod
            ks.append((len(ks) + 1, h))
        keys.append(ks)
        for key in ks:
            cnt[key] += 1
    maxlen = max(map(len, words))
    good = [0] * (maxlen + 1)
    for (d, _), v in cnt.items():
        if v >= k:
            good[d] += 1
    maxD = max((d for d in range(maxlen + 1) if good[d] > 0), default=0)
    res = []
    for ks in keys:
        d = maxD
        while d > 0:
            lose = 1 if d <= len(ks) and cnt[ks[d - 1]] == k else 0
            if good[d] - lose > 0:
                break
            d -= 1
        res.append(d)
    return res

for w, k, exp in examples:
    print(lcpk_hash(w, k), '| kỳ vọng', exp)

[3, 4, 4, 3, 4] | kỳ vọng [3, 4, 4, 3, 4]
[0, 0, 0] | kỳ vọng [0, 0, 0]


## 8. TRACE từng bước

### 8.1. Trace cách 2 trên ví dụ 1

In [11]:
def trace_sorted(words, k):
    n = len(words)
    order = sorted(range(n), key=lambda i: words[i])
    s = [words[i] for i in order]
    print('sau khi sắp:', s, '| order =', order)
    A = [lcp(s[j], s[j + k - 1]) for j in range(n - k + 1)]
    B = [lcp(s[j], s[j + k]) for j in range(n - k)]
    print('A (cửa sổ k)   =', A)
    print('B (cửa sổ k+1) =', B)
    for p, i in enumerate(order):
        left = A[:max(0, p - k + 1)]
        right = A[p + 1:]
        mid = B[max(0, p - k):min(p, len(B) - 1) + 1]
        best = max(left + right + mid, default=0)
        print(f"bỏ words[{i}]={words[i]!r} (vị trí {p}): A trái {left}, A phải {right}, B vắt qua {mid} -> {best}")

trace_sorted(*examples[0][:2])

sau khi sắp: ['jump', 'jump', 'run', 'run', 'run'] | order = [0, 3, 1, 2, 4]
A (cửa sổ k)   = [4, 0, 3, 3]
B (cửa sổ k+1) = [0, 0, 3]
bỏ words[0]='jump' (vị trí 0): A trái [], A phải [0, 3, 3], B vắt qua [0] -> 3
bỏ words[3]='jump' (vị trí 1): A trái [], A phải [3, 3], B vắt qua [0, 0] -> 3
bỏ words[1]='run' (vị trí 2): A trái [4], A phải [3], B vắt qua [0, 0, 3] -> 4
bỏ words[2]='run' (vị trí 3): A trái [4, 0], A phải [], B vắt qua [0, 3] -> 4
bỏ words[4]='run' (vị trí 4): A trái [4, 0, 3], A phải [], B vắt qua [3] -> 4


### 8.2. Trace cách 3 (trie)

In [12]:
def trace_trie(words, k):
    children, cnt, depth, paths = build_trie(words)
    maxlen = max(map(len, words))
    good = [0] * (maxlen + 1)
    for u in range(1, len(cnt)):
        if cnt[u] >= k:
            good[depth[u]] += 1
    maxD = max((d for d in range(maxlen + 1) if good[d] > 0), default=0)
    print('good theo độ sâu:', good, '| maxD =', maxD)
    for i, (w, path) in enumerate(zip(words, paths)):
        d, steps = maxD, []
        while d > 0:
            lose = 1 if d <= len(path) and cnt[path[d - 1]] == k else 0
            steps.append(f"d={d}: good={good[d]}, mất={lose}")
            if good[d] - lose > 0:
                break
            d -= 1
        print(f"bỏ {w!r}: " + '; '.join(steps) + f" -> {d}")

trace_trie(*examples[0][:2])

good theo độ sâu: [0, 2, 2, 2, 1] | maxD = 4
bỏ 'jump': d=4: good=1, mất=1; d=3: good=2, mất=1 -> 3
bỏ 'run': d=4: good=1, mất=0 -> 4
bỏ 'run': d=4: good=1, mất=0 -> 4
bỏ 'jump': d=4: good=1, mất=1; d=3: good=2, mất=1 -> 3
bỏ 'run': d=4: good=1, mất=0 -> 4


## 9. Code nộp LeetCode

Dùng cách 3 (trie) — ngắn, tuyến tính theo tổng độ dài.

In [13]:
class Solution:
    def longestCommonPrefix(self, words: List[str], k: int) -> List[int]:
        n = len(words)
        if n - 1 < k:
            return [0] * n
        children = [{}]
        cnt = [0]
        depth = [0]
        paths = []
        for w in words:
            u = 0
            path = []
            for c in w:
                v = children[u].get(c)
                if v is None:
                    v = len(children)
                    children[u][c] = v
                    children.append({})
                    cnt.append(0)
                    depth.append(depth[u] + 1)
                u = v
                cnt[u] += 1
                path.append(u)
            paths.append(path)
        maxlen = max(map(len, words))
        good = [0] * (maxlen + 1)
        for u in range(1, len(cnt)):
            if cnt[u] >= k:
                good[depth[u]] += 1
        maxD = 0
        for d in range(maxlen, 0, -1):
            if good[d] > 0:
                maxD = d
                break
        ans = []
        for path in paths:
            d = maxD
            while d > 0:
                lose = 1 if d <= len(path) and cnt[path[d - 1]] == k else 0
                if good[d] - lose > 0:
                    break
                d -= 1
            ans.append(d)
        return ans

for w, k, exp in examples:
    got = Solution().longestCommonPrefix(w, k)
    print(got, 'OK' if got == exp else 'SAI')

[3, 4, 4, 3, 4] OK
[0, 0, 0] OK


## 10. Vì sao thuật toán đúng

**Cách 3 (trie).**

1. Sau khi bỏ từ `w`, đếm mới của nút `u` là `cnt[u] − [u nằm trên đường đi của w]`. Nút tốt mới ⇔ đếm mới ≥ `k`.
2. Nút ở độ sâu `d` trên đường của `w` là **duy nhất** (`path[d − 1]`). Nó mất tư cách ⇔ `cnt = k`. Các nút khác ở độ sâu `d` không đổi. ⇒ số nút tốt mới ở độ sâu `d` là `good[d] − lose(d)`.
3. Đáp án là `d` lớn nhất có số nút tốt mới > 0. Mọi `d > maxD` có `good[d] = 0` nên bắt đầu dò từ `maxD`. Dò xuống tới khi gặp `d` thỏa ⇒ chính là `d` lớn nhất.
4. `n − 1 < k` ⇒ không chọn nổi `k` chuỗi ⇒ `0`.

**Cách 2 (sắp xếp).**

1. Mọi nhóm `k` chuỗi có tiền tố chung `P` ⇒ trong danh sách đã sắp có ≥ `k` chuỗi liên tiếp mang `P` ⇒ tồn tại cửa sổ liên tiếp có LCP ≥ `|P|`. Ngược lại LCP của cửa sổ là LCP của một nhóm hợp lệ. ⇒ đáp án = max LCP cửa sổ `k` phần tử **liên tiếp trong danh sách còn lại**.
2. Danh sách còn lại sau khi bỏ `p` vẫn đã sắp. Mỗi cửa sổ `k` phần tử liên tiếp của nó hoặc là cửa sổ gốc không chứa `p` (giá trị `A[j]`), hoặc là `[j, j+k] \ {p}` với `j < p < j + k` (giá trị `lcp(s[j], s[j+k]) = B[j]`). Lấy `j ∈ [p − k, p]` rộng hơn một chút vẫn đúng vì với `j = p − k` hay `j = p` thì `B[j]` ≤ giá trị của một cửa sổ `k` phần tử không chứa `p` đã có trong mục 1.

## 11. Phân tích Time / Space Complexity

Gọi `S = Σ|w| ≤ 10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. Đếm lại | `O(n · Σ|w|²)` | `O(Σ|w|²)` |
| 2. Sắp xếp + sparse table | `O(S log n + n log n)` (so sánh chuỗi khi sắp; LCP tổng `O(S)`) | `O(n log n + S)` |
| 3. Trie | `O(S)` | `O(S)` nút |
| 4. Băm tiền tố | `O(S)` kỳ vọng | `O(S)` |

Ghi chú: tổng các `lcp(s[j], s[j + k − 1])` bị chặn bởi `Σ|s[j]|` ≤ `S`, và tương tự cho `B`.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [14]:
impls = {'sorted': lcpk_sorted, 'trie': lcpk_trie, 'hash': lcpk_hash,
         'submit': lambda w, k: Solution().longestCommonPrefix(w, k)}
edge = [
    (["a"], 1, [0]),                              # bỏ xong không còn chuỗi nào
    (["abc", "abd"], 2, [0, 0]),                  # n - 1 < k
    (["abc", "abd"], 1, [3, 3]),                  # k = 1: chuỗi dài nhất còn lại
    (["aa", "aa", "aa"], 3, [0, 0, 0]),
    (["ab", "ab", "a", "a"], 2, [1, 1, 2, 2]),         # bỏ một "ab" thì "ab" chỉ còn 1
    (["abcd", "abcd", "abce", "x"], 2, [3, 3, 4, 4]),
]
for w, k, exp in examples + edge:
    assert lcpk_brute(w, k) == exp, (w, k, lcpk_brute(w, k))
    for name, f in impls.items():
        assert f(w, k) == exp, (name, w, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [15]:
import random
rng = random.Random(3485)
cnt_tests = 0
for _ in range(2000):
    n = rng.randint(1, 9)
    k = rng.randint(1, n)
    alpha = rng.choice(['ab', 'abc', 'a'])
    w = [''.join(rng.choice(alpha) for _ in range(rng.randint(1, 5))) for _ in range(n)]
    exp = lcpk_brute(w, k)
    for name, f in impls.items():
        assert f(w, k) == exp, (name, w, k)
    cnt_tests += 1
print('Khớp vét cạn trên', cnt_tests, 'bộ ngẫu nhiên.')

Khớp vét cạn trên 2000 bộ ngẫu nhiên.


### 12.3. Ca lớn (tổng độ dài `10^5`)

In [16]:
import time
rng = random.Random(7)
big_cases = {
    '10^5 chuỗi 1 ký tự': [rng.choice('ab') for _ in range(10 ** 5)],
    '10 chuỗi dài 10^4 giống nhau': ['a' * 10 ** 4] * 10,
    '5000 chuỗi dài 20': [''.join(rng.choice('ab') for _ in range(20)) for _ in range(5000)],
    '2 nhóm dài 5·10^4': ['a' * 49999 + 'b', 'a' * 49999 + 'c'],
}
for name, w in big_cases.items():
    for k in [1, 2, max(1, len(w) // 2)]:
        t = time.time(); r_sub = Solution().longestCommonPrefix(w, k); t1 = time.time() - t
        t = time.time(); r_sort = lcpk_sorted(w, k); t2 = time.time() - t
        assert r_sub == r_sort
        print(f"{name:30s} k={k:6d}: max={max(r_sub)} | trie {t1:.2f}s | sắp xếp {t2:.2f}s")

10^5 chuỗi 1 ký tự             k=     1: max=1 | trie 0.03s | sắp xếp 0.18s
10^5 chuỗi 1 ký tự             k=     2: max=1 | trie 0.05s | sắp xếp 0.15s


10^5 chuỗi 1 ký tự             k= 50000: max=1 | trie 0.03s | sắp xếp 0.10s
10 chuỗi dài 10^4 giống nhau   k=     1: max=10000 | trie 0.01s | sắp xếp 0.01s
10 chuỗi dài 10^4 giống nhau   k=     2: max=10000 | trie 0.01s | sắp xếp 0.01s
10 chuỗi dài 10^4 giống nhau   k=     5: max=10000 | trie 0.01s | sắp xếp 0.00s
5000 chuỗi dài 20              k=     1: max=20 | trie 0.03s | sắp xếp 0.01s
5000 chuỗi dài 20              k=     2: max=20 | trie 0.03s | sắp xếp 0.01s
5000 chuỗi dài 20              k=  2500: max=1 | trie 0.05s | sắp xếp 0.00s
2 nhóm dài 5·10^4              k=     1: max=50000 | trie 0.02s | sắp xếp 0.00s
2 nhóm dài 5·10^4              k=     2: max=0 | trie 0.00s | sắp xếp 0.00s


2 nhóm dài 5·10^4              k=     1: max=50000 | trie 0.02s | sắp xếp 0.00s


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ nhỏ nhất (theo seed cố định) để minh họa:

In [17]:
def find_counterexample(wrong, seed=0, tries=20000):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 6)
        k = rng.randint(1, n)
        w = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 3))) for _ in range(n)]
        if wrong(w, k) != lcpk_brute(w, k):
            return w, k
    return None

### Lỗi 1 — Tính đáp án một lần, không quan tâm phần tử bị bỏ

In [18]:
def wrong_global(words, k):
    n = len(words)
    if n - 1 < k:
        return [0] * n
    c = prefix_counts(words)
    g = max([len(p) for p, v in c.items() if v >= k], default=0)
    return [g] * n

w, k, _ = examples[0]
print('sai :', wrong_global(w, k), '| đúng:', Solution().longestCommonPrefix(w, k))

sai : [4, 4, 4, 4, 4] | đúng: [3, 4, 4, 3, 4]


**Vì sao sai:** bỏ một `"jump"` thì `"jump"` chỉ còn 1 lần ⇒ không đủ `k = 2`.

### Lỗi 2 — Cách sắp xếp nhưng quên các cửa sổ "vắt qua" phần tử bị bỏ (mảng `B`)

In [19]:
def wrong_no_B(words, k):
    n = len(words)
    if n - 1 < k:
        return [0] * n
    order = sorted(range(n), key=lambda i: words[i])
    s = [words[i] for i in order]
    pos = {i: p for p, i in enumerate(order)}
    A = [lcp(s[j], s[j + k - 1]) for j in range(n - k + 1)]
    res = []
    for i in range(n):
        p = pos[i]
        res.append(max(A[:max(0, p - k + 1)] + A[p + 1:], default=0))   # SAI: thiếu B
    return res

ce = find_counterexample(wrong_no_B, seed=2)
print('phản ví dụ:', ce)
print('sai :', wrong_no_B(*ce), '| đúng:', lcpk_brute(*ce))

phản ví dụ: (['ba', 'bb', 'bbb', 'bb'], 2)
sai : [2, 2, 2, 1] | đúng: [2, 2, 2, 2]


**Vì sao sai:** bỏ `p` làm hai phần tử hai bên `p` trở thành **kề nhau**, tạo cửa sổ mới chưa từng có trong `A`. Ví dụ `["a","a","a"]` với `k = 2` (sắp rồi), bỏ phần tử giữa: hai chữ `a` còn lại vẫn có LCP 1, nhưng mọi cửa sổ cũ đều chứa phần tử giữa.

### Lỗi 3 — Trie: trừ đi khi `cnt ≥ k` thay vì `cnt == k`

In [20]:
def wrong_lose_ge(words, k):
    n = len(words)
    if n - 1 < k:
        return [0] * n
    children, cnt, depth, paths = build_trie(words)
    maxlen = max(map(len, words))
    good = [0] * (maxlen + 1)
    for u in range(1, len(cnt)):
        if cnt[u] >= k:
            good[depth[u]] += 1
    maxD = max((d for d in range(maxlen + 1) if good[d] > 0), default=0)
    res = []
    for path in paths:
        d = maxD
        while d > 0:
            lose = 1 if d <= len(path) and cnt[path[d - 1]] >= k else 0   # SAI
            if good[d] - lose > 0:
                break
            d -= 1
        res.append(d)
    return res

ce = find_counterexample(wrong_lose_ge, seed=3)
print('phản ví dụ:', ce)
print('sai :', wrong_lose_ge(*ce), '| đúng:', lcpk_brute(*ce))

phản ví dụ: (['bb', 'bbb', 'b'], 1)
sai : [3, 0, 3] | đúng: [3, 2, 3]


**Vì sao sai:** nút có `cnt > k` sau khi bỏ một từ vẫn còn `≥ k` ⇒ vẫn tốt. Chỉ nút có `cnt` **đúng bằng** `k` mới rơi xuống `k − 1`.

### Lỗi 4 — Xét cửa sổ liên tiếp theo **thứ tự gốc** (quên sắp xếp)

In [21]:
def wrong_unsorted(words, k):
    n = len(words)
    res = []
    for i in range(n):
        rest = words[:i] + words[i + 1:]
        res.append(max((lcp_group(rest[j:j + k]) for j in range(len(rest) - k + 1)), default=0))
    return res

w, k, _ = examples[0]
print('sai :', wrong_unsorted(w, k), '| đúng:', Solution().longestCommonPrefix(w, k))

sai : [3, 0, 0, 3, 3] | đúng: [3, 4, 4, 3, 4]


**Vì sao sai:** `k` chuỗi được chọn ở **bất kỳ** chỉ số nào, không cần liền nhau trong mảng gốc. Chỉ sau khi sắp xếp mới có tính chất "nhóm tốt nhất là liên tiếp".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tiền tố chung của ít nhất `k` chuỗi" ⇒ trie với `cnt` ≥ `k`, hoặc sắp xếp + cửa sổ `k`.
- "Đáp án cho mỗi `i` khi bỏ phần tử `i`" ⇒ tiền tố/hậu tố max + xử lý riêng phần bị ảnh hưởng (ở đây: cửa sổ `k + 1`, hoặc các nút `cnt == k`).
- Bỏ một phần tử chỉ làm giảm đếm **1** ⇒ ngưỡng quan trọng là `cnt == k` (đúng ngưỡng).
- Tổng độ dài `≤ 10^5` ⇒ mọi thứ tuyến tính theo tổng ký tự là đủ.

## 15. Biến thể của bài

1. **LC 14 — Longest Common Prefix:** tất cả chuỗi (k = n), không bỏ.
2. **LC 3043 — Longest Common Prefix giữa hai mảng số:** trie trên chữ số.
3. Bỏ **hai** phần tử ⇒ nút mất tư cách nếu `cnt − (số từ bị bỏ đi qua nó) < k`.
4. Thay "bỏ" bằng "thêm" một chuỗi mới ⇒ nút có `cnt == k − 1` trên đường đi trở thành tốt.

In [22]:
# Biến thể 4: với mỗi truy vấn chuỗi q, đáp án nếu THÊM q vào mảng
def lcpk_after_add(words, k, queries):
    children, cnt, depth, _ = build_trie(words)
    maxlen = max(map(len, words + queries))
    good = [0] * (maxlen + 1)
    for u in range(1, len(cnt)):
        if cnt[u] >= k:
            good[depth[u]] += 1
    base = max((d for d in range(maxlen + 1) if good[d] > 0), default=0)
    res = []
    for q in queries:
        best, u = base, 0
        for d, c in enumerate(q, 1):
            u = children[u].get(c)
            if u is None:
                if k == 1:
                    best = max(best, len(q))   # chính q tạo tiền tố mới
                break
            if cnt[u] + 1 >= k:
                best = max(best, d)
        else:
            pass
        res.append(best)
    return res

def add_brute(words, k, queries):
    out = []
    for q in queries:
        c = prefix_counts(words + [q])
        out.append(max([len(p) for p, v in c.items() if v >= k], default=0))
    return out

rng = random.Random(15)
for _ in range(500):
    w = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 6))]
    qs = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 4))) for _ in range(3)]
    k = rng.randint(1, len(w) + 1)
    assert lcpk_after_add(w, k, qs) == add_brute(w, k, qs), (w, k, qs)
print('Biến thể 4 khớp vét cạn.', lcpk_after_add(['jump', 'run'], 2, ['jumbo', 'rx', 'zz']))

Biến thể 4 khớp vét cạn. [3, 1, 0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Đếm lại tiền tố | `O(n Σ|w|²)` | Oracle |
| 2 | Sắp xếp + cửa sổ `k`, `k+1` + sparse table | `O(S log n)` | Không cần trie |
| 3 | Trie + `good[d]` | `O(S)` | **Nộp bài** |
| 4 | Băm tiền tố | `O(S)` kỳ vọng | Có xác suất va chạm rất nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `words = ["abc","abd","ab"], k = 2`.

**Bài 2 (tính tay):** `words = ["aaa","aa","a","aaa"], k = 3`.

**Bài 3 (điền TODO):** viết `lcp_k_no_removal(words, k)` — đáp án khi **không** bỏ phần tử nào, bằng sắp xếp.

**Bài 4 (tự viết lại):** thay sparse table ở cách 2 bằng **max cửa sổ trượt** (deque) vì đoạn truy vấn luôn dài `k + 1`.

**Bài 5 (mini challenge):** biến thể 3 — bỏ **hai** chỉ số `i < j` (cho danh sách cặp truy vấn).

In [23]:
def lcp_k_no_removal_todo(words, k):
    # TODO
    raise NotImplementedError

def lcpk_sorted_deque_todo(words, k):
    # TODO
    raise NotImplementedError

def remove_two_todo(words, k, pairs):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tiền tố `ab` có 3 chuỗi. (2) Bỏ một chuỗi vẫn còn 2. (3) `"abc"`, `"abd"` không chung độ dài 3.

**Bài 2.** (1) Đếm: `a`:4, `aa`:3, `aaa`:2. (2) Bỏ chuỗi có `aa` (đếm đúng bằng `k = 3`) ⇒ tụt về 1. (3) Bỏ `"a"` thì `aa` vẫn 3.

**Bài 3.** (1) Sắp xếp. (2) `max lcp(s[j], s[j + k − 1])`. (3) `n < k` ⇒ 0.

**Bài 4.** (1) Chỉ số truy vấn `[p − k, p]` tăng dần theo `p`. (2) Deque giữ chỉ số có `B` giảm dần. (3) Kẹp biên `0` và `len(B) − 1`.

**Bài 5.** (1) Với trie: tại độ sâu `d`, hai đường đi có thể trùng nút. (2) Nút mất tư cách nếu `cnt − số từ bị bỏ đi qua < k`. (3) Dò từ `maxD` xuống, chỉ xét tối đa hai nút ở mỗi độ sâu.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
a1 = Solution().longestCommonPrefix(["abc", "abd", "ab"], 2)
a2 = Solution().longestCommonPrefix(["aaa", "aa", "a", "aaa"], 3)
assert a1 == lcpk_brute(["abc", "abd", "ab"], 2) == [2, 2, 2]
assert a2 == lcpk_brute(["aaa", "aa", "a", "aaa"], 3) == [1, 1, 2, 1]
print('Bài 1:', a1, '| Bài 2:', a2)

# Bài 3
def lcp_k_no_removal(words, k):
    if len(words) < k:
        return 0
    s = sorted(words)
    return max(lcp(s[j], s[j + k - 1]) for j in range(len(s) - k + 1))

for w, k, _ in examples:
    assert lcp_k_no_removal(w, k) == max([len(p) for p, v in prefix_counts(w).items() if v >= k], default=0)
print('Bài 3:', [lcp_k_no_removal(w, k) for w, k, _ in examples])

Bài 1: [2, 2, 2] | Bài 2: [1, 1, 2, 1]
Bài 3: [4, 0]


In [25]:
# Bài 4 — deque
from collections import deque

def lcpk_sorted_deque(words, k):
    n = len(words)
    if n - 1 < k:
        return [0] * n
    order = sorted(range(n), key=lambda i: words[i])
    s = [words[i] for i in order]
    A = [lcp(s[j], s[j + k - 1]) for j in range(n - k + 1)]
    B = [lcp(s[j], s[j + k]) for j in range(n - k)]
    pre = [0] * (len(A) + 1)
    for j, x in enumerate(A):
        pre[j + 1] = max(pre[j], x)
    suf = [0] * (len(A) + 1)
    for j in range(len(A) - 1, -1, -1):
        suf[j] = max(suf[j + 1], A[j])
    ans = [0] * n
    dq, nxt = deque(), 0
    for p in range(n):                       # duyệt vị trí trong s theo thứ tự tăng
        hi = min(p, len(B) - 1)
        while nxt <= hi:
            while dq and B[dq[-1]] <= B[nxt]:
                dq.pop()
            dq.append(nxt); nxt += 1
        lo = max(0, p - k)
        while dq and dq[0] < lo:
            dq.popleft()
        mid = B[dq[0]] if dq else 0
        ans[order[p]] = max(pre[max(0, p - k + 1)], suf[min(len(A), p + 1)], mid)
    return ans

rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 9); k = rng.randint(1, n)
    w = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 4))) for _ in range(n)]
    assert lcpk_sorted_deque(w, k) == lcpk_brute(w, k)
print('Bài 4 khớp vét cạn.')

Bài 4 khớp vét cạn.


In [26]:
# Bài 5 — bỏ hai chỉ số
def remove_two(words, k, pairs):
    n = len(words)
    children, cnt, depth, paths = build_trie(words)
    maxlen = max(map(len, words))
    good = [0] * (maxlen + 1)
    for u in range(1, len(cnt)):
        if cnt[u] >= k:
            good[depth[u]] += 1
    maxD = max((d for d in range(maxlen + 1) if good[d] > 0), default=0)
    out = []
    for i, j in pairs:
        if n - 2 < k:
            out.append(0); continue
        d = maxD
        while d > 0:
            hit = Counter()
            for path in (paths[i], paths[j]):
                if d <= len(path):
                    hit[path[d - 1]] += 1
            lose = sum(1 for u, h in hit.items() if cnt[u] >= k and cnt[u] - h < k)
            if good[d] - lose > 0:
                break
            d -= 1
        out.append(d)
    return out

def remove_two_brute(words, k, pairs):
    out = []
    for i, j in pairs:
        rest = [w for t, w in enumerate(words) if t not in (i, j)]
        out.append(max([len(p) for p, v in prefix_counts(rest).items() if v >= k], default=0))
    return out

rng = random.Random(5)
for _ in range(400):
    n = rng.randint(2, 8); k = rng.randint(1, n)
    w = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 4))) for _ in range(n)]
    pairs = [tuple(sorted(rng.sample(range(n), 2))) for _ in range(3)]
    assert remove_two(w, k, pairs) == remove_two_brute(w, k, pairs), (w, k, pairs)
print('Bài 5 khớp vét cạn.', remove_two(["jump", "run", "run", "jump", "run"], 2, [(0, 3), (1, 2)]))

Bài 5 khớp vét cạn. [3, 4]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Trie: cnt[u] = số từ đi qua u; good[d] = #nút độ sâu d có cnt >= k
maxD = độ sâu lớn nhất có good > 0
for path in paths:                     # path[d-1] = nút độ sâu d của từ
    d = maxD
    while d > 0 and good[d] - (d <= len(path) and cnt[path[d-1]] == k) == 0:
        d -= 1
    ans.append(d)
# n - 1 < k  =>  toàn 0
```

**Checklist:**

- [ ] `n − 1 < k` ⇒ trả toàn `0`?
- [ ] Chỉ nút có `cnt == k` trên đường đi của từ bị bỏ mới mất tư cách?
- [ ] Bắt đầu dò từ `maxD` (không phải `len(word)`)?
- [ ] Nếu dùng sắp xếp: có tính các cửa sổ `k + 1` vắt qua phần tử bị bỏ?
- [ ] Không dùng đệ quy trên trie sâu `10^4`?